# Heatmaps: Topicos Sustantivos - Camara de Diputados (2008-2025)

Analisis de prevalencia de los 11 topicos sustantivos del modelo NMF K=24 (P1):
- Por anio
- Por periodo presidencial
- Por evento historico


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

DATA_DIR = Path(r'C:\Users\Saroka\Documents\GitHub\repositorios\argentine-deputies-discursive-distance\data\qa\topic_modeling\selected_nmf_k024_v1')
FIG_DIR = Path('figuras/exp2')
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
TOPICOS_SUSTANTIVOS = {
    1:  'Derecho penal y codigos procesales',
    4:  'Justicia y poder judicial',
    5:  'Provincias, Buenos Aires y territorio federal',
    8:  'Desarrollo productivo y economia',
    9:  'Derechos, genero y salud reproductiva',
    10: 'Trabajo y empleo',
    16: 'Poderes del ejecutivo, DNU y autoridad constitucional',
    17: 'Presupuesto, gasto e inflacion',
    18: 'Jubilaciones y seguridad social',
    19: 'Deuda, FMI y politica financiera',
    20: 'Impuestos y politica fiscal',
}
TOPICOS_IDS = list(TOPICOS_SUSTANTIVOS.keys())
ORDEN_TOPICOS = [TOPICOS_SUSTANTIVOS[i] for i in TOPICOS_IDS]
print(f'{len(TOPICOS_IDS)} topicos sustantivos seleccionados')


11 topicos sustantivos seleccionados


## 2. Heatmap por anio


In [3]:
annual = pd.read_csv(DATA_DIR / 'annual_topic_prevalence.csv')
df_anual = annual[
    (annual['aggregation_level'] == 'source_turn') &
    (annual['topic_index'].isin(TOPICOS_IDS))
].copy()
df_anual['topico'] = df_anual['topic_index'].map(TOPICOS_SUSTANTIVOS)
pivot_anual = df_anual.pivot(index='topico', columns='year', values='prevalence_share').reindex(ORDEN_TOPICOS)

fig_anual = px.imshow(pivot_anual,
    labels=dict(x='Anio', y='Topico', color='Prevalencia'),
    title='Prevalencia de topicos sustantivos por anio (2008-2025)',
    color_continuous_scale='Blues', aspect='auto', height=550, text_auto='.2%')
fig_anual.update_layout(coloraxis_colorbar=dict(tickformat='.0%'))
fig_anual.show()
fig_anual.write_html(str(FIG_DIR / 'heatmap_topicos_por_anio.html'))


## 3. Heatmap por periodo presidencial


In [4]:
period = pd.read_csv(DATA_DIR / 'period_topic_prevalence.csv')
df_periodo = period[
    (period['aggregation_level'] == 'source_turn') &
    (period['topic_index'].isin(TOPICOS_IDS))
].copy()
df_periodo['topico'] = df_periodo['topic_index'].map(TOPICOS_SUSTANTIVOS)

PERIODO_LABELS = {
    '2008-2011': 'CFK I (2008-2011)',
    '2012-2015': 'CFK II (2012-2015)',
    '2016-2019': 'Macri (2016-2019)',
    '2020-2023': 'Fernandez (2020-2023)',
    '2024-2025': 'Milei (2024-2025)',
}
df_periodo['periodo_label'] = df_periodo['temporal_period'].map(PERIODO_LABELS)
pivot_periodo = df_periodo.pivot(index='topico', columns='periodo_label', values='prevalence_share').reindex(ORDEN_TOPICOS)
pivot_periodo = pivot_periodo[[v for v in PERIODO_LABELS.values() if v in pivot_periodo.columns]]

fig_periodo = px.imshow(pivot_periodo,
    labels=dict(x='Periodo presidencial', y='Topico', color='Prevalencia'),
    title='Prevalencia de topicos sustantivos por periodo presidencial',
    color_continuous_scale='Blues', aspect='auto', height=550, text_auto='.2%')
fig_periodo.update_layout(coloraxis_colorbar=dict(tickformat='.0%'), xaxis_tickangle=30)
fig_periodo.show()
fig_periodo.write_html(str(FIG_DIR / 'heatmap_topicos_por_periodo_presidencial.html'))


## 4. Heatmap por evento historico

Ventana: anio del evento + anio siguiente.


In [5]:
metadata = pd.read_csv(DATA_DIR / 'source_turn_metadata.csv')
weights = np.load(DATA_DIR / 'topic_weights.npy')
df_turns = metadata[['source_turn_row_index', 'year']].copy()
for tid in TOPICOS_IDS:
    df_turns[f'topic_{tid}'] = weights[:, tid]
print(f'Source turns cargados: {len(df_turns):,}')


Source turns cargados: 34,060


In [6]:
EVENTOS_HISTORICOS = {
    2008: 'Res. 125',
    2010: 'Mat. Igualitario',
    2012: 'YPF',
    2015: 'Muerte Nisman',
    2018: 'IVE',
    2019: 'Reperfilamiento',
    2020: 'COVID-19',
    2023: 'Ley Bases / DNU',
}

filas = []
for anio_evento, nombre in EVENTOS_HISTORICOS.items():
    ventana = df_turns[df_turns['year'].isin([anio_evento, anio_evento + 1])]
    if len(ventana) == 0:
        continue
    label = f'{nombre} ({anio_evento}-{anio_evento + 1})'
    for tid in TOPICOS_IDS:
        filas.append({'evento': label, 'topico': TOPICOS_SUSTANTIVOS[tid],
                      'prevalencia': ventana[f'topic_{tid}'].mean()})

df_eventos = pd.DataFrame(filas)
orden_eventos = [f'{n} ({a}-{a+1})' for a, n in sorted(EVENTOS_HISTORICOS.items())]
pivot_eventos = df_eventos.pivot(index='topico', columns='evento', values='prevalencia').reindex(ORDEN_TOPICOS)
pivot_eventos = pivot_eventos[[c for c in orden_eventos if c in pivot_eventos.columns]]

fig_eventos = px.imshow(pivot_eventos,
    labels=dict(x='Evento historico', y='Topico', color='Prevalencia'),
    title='Prevalencia de topicos sustantivos por evento historico',
    color_continuous_scale='Reds', aspect='auto', height=550, text_auto='.2%')
fig_eventos.update_layout(coloraxis_colorbar=dict(tickformat='.0%'), xaxis_tickangle=30)
fig_eventos.show()
fig_eventos.write_html(str(FIG_DIR / 'heatmap_topicos_por_evento_historico.html'))
